# 04.01 — NetworkX Backend

Orthograph provides a two-phase architecture for understanding graph data:

1. **Inspect** -- a `GraphInspector` scans a graph source and produces a `GraphProfile`: a structural summary of node types, relationship types, property completeness, observed types, and cardinality statistics.
2. **Compare** -- the `compare` function compares a `GraphProfile` against a `GraphDefinition` and reports mismatches.

This notebook demonstrates the full workflow using **NetworkX** as the graph backend, with the same filmography domain (Person, Movie, City) used throughout these notebooks.

In [1]:
import networkx as nx

from orthograph.backends.networkx.conversion import schema_to_networkx
from orthograph.compare import profile_to_definition
from orthograph.profile import inspect_networkx

## Define the model

The familiar filmography model: people act in and direct movies, and live in cities.

In [2]:
from shared.filmography import FILMOGRAPHY_MODEL


graph_definition = FILMOGRAPHY_MODEL
print("Model:", graph_definition.name, "| nodes:", sorted(graph_definition.node_labels))

Model: Filmography | nodes: ['City', 'Movie', 'Person']


## Build a NetworkX graph with sample data

We populate a `MultiDiGraph` with people, movies, and cities. To demonstrate property completeness analysis, we intentionally:

- Omit the `email` property on all Person nodes (optional, so this is fine)
- Omit the `age` property on one Person node (required -- this will trigger a warning)

In [3]:
G = nx.MultiDiGraph()

# People -- note p3 is missing the 'age' property
G.add_node("p1", __label__="Person", name="Alice", age=30, email="alice@example.com")
G.add_node("p2", __label__="Person", name="Bob", age=45)
G.add_node("p3", __label__="Person", name="Charlie")  # missing 'age'

# Movies
G.add_node("m1", __label__="Movie", title="The Matrix", year=1999, rating=8.7)
G.add_node("m2", __label__="Movie", title="Inception", year=2010)

# Cities
G.add_node("c1", __label__="City", name="Los Angeles", country="USA")

# Relationships
G.add_edge("p1", "m1", __label__="ACTED_IN", role="Trinity")
G.add_edge("p2", "m1", __label__="ACTED_IN", role="Morpheus")
G.add_edge("p1", "m2", __label__="ACTED_IN", role="Ariadne")
G.add_edge("p2", "m2", __label__="DIRECTED")
G.add_edge("p1", "c1", __label__="LIVES_IN")
G.add_edge("p2", "c1", __label__="LIVES_IN")

print(f"Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

Graph has 6 nodes and 6 edges


## Inspect the graph

`NetworkxInspector` scans the graph and produces a `GraphProfile` -- a frozen Pydantic model that summarizes everything about the graph's structure. No model definition is needed at this stage; the inspector reports what it finds.

In [4]:
# inspect_networkx imported above
profile = inspect_networkx(G)

print("Profile source:   ", profile.source)
print("Profile timestamp:", profile.timestamp)
print("Node labels:      ", profile.node_labels)
print("Relationship types:", profile.relationship_types)

Profile source:    networkx
Profile timestamp: 2026-06-27 03:04:08.046213
Node labels:       {'Person', 'City', 'Movie'}
Relationship types: {'Person:ACTED_IN:Movie', 'Person:DIRECTED:Movie', 'Person:LIVES_IN:City'}


## Explore node type profiles

Each `NodeTypeProfile` contains the label, instance count, and a `PropertyProfile` for every property observed across all instances of that type. The property profile tracks completeness (what fraction of nodes have the property) and observed Python types.

In [5]:
for label, ntp in profile.node_type_profiles.items():
    print(f"--- {label} ({ntp.count} instances) ---")
    for prop_name, pp in ntp.property_profiles.items():
        print(
            f"  {prop_name:12s}  "
            f"completeness={pp.completeness:.0%}  "
            f"({pp.present_count}/{pp.total_count})  "
            f"types={pp.observed_types}"
        )
    print()

--- City (1 instances) ---
  country       completeness=100%  (1/1)  types=['str']
  name          completeness=100%  (1/1)  types=['str']

--- Movie (2 instances) ---
  rating        completeness=50%  (1/2)  types=['float']
  title         completeness=100%  (2/2)  types=['str']
  year          completeness=100%  (2/2)  types=['int']

--- Person (3 instances) ---
  age           completeness=67%  (2/3)  types=['int']
  email         completeness=33%  (1/3)  types=['str']
  name          completeness=100%  (3/3)  types=['str']



Notice that:

- `age` on `Person` has only 67% completeness -- `Charlie` is missing it
- `email` on `Person` has only 33% completeness -- only `Alice` has it
- `rating` on `Movie` has 50% completeness -- only `The Matrix` has it

The profile captures this factually. Whether these are *problems* depends on the model definition -- `email` and `rating` are optional, so their low completeness is expected. But `age` is required, so that's a data quality issue.

## Explore relationship type profiles

Each `RelationshipTypeProfile` contains the relationship type, count, source/target labels, property profiles, and cardinality statistics (min/max/avg outgoing degree from source nodes).

In [6]:
for rel_type, rtp in profile.rel_type_profiles.items():
    print(f"--- {rel_type} ({rtp.count} instances) ---")
    print(f"  source labels: {rtp.source_label}")
    print(f"  target labels: {rtp.target_label}")
    if rtp.cardinality_stats:
        cs = rtp.cardinality_stats
        print(
            f"  cardinality:   min={cs.min}, max={cs.max}, "
            f"avg={cs.mean:.1f}, sample_size={cs.count}"
        )
    for prop_name, pp in rtp.property_profiles.items():
        print(
            f"  {prop_name:12s}  "
            f"completeness={pp.completeness:.0%}  "
            f"types={pp.observed_types}"
        )
    print()

--- Person:ACTED_IN:Movie (3 instances) ---
  source labels: Person
  target labels: Movie
  cardinality:   min=1.0, max=2.0, avg=1.5, sample_size=2
  role          completeness=100%  types=['str']

--- Person:DIRECTED:Movie (1 instances) ---
  source labels: Person
  target labels: Movie
  cardinality:   min=1.0, max=1.0, avg=1.0, sample_size=1

--- Person:LIVES_IN:City (2 instances) ---
  source labels: Person
  target labels: City
  cardinality:   min=1.0, max=1.0, avg=1.0, sample_size=2



## Compare the profile against the model

`compare` compares the `GraphProfile` against the `GraphDefinition` and reports:

- **Errors**: missing required types, type mismatches, invalid endpoints, cardinality violations
- **Warnings**: incomplete required properties, unexpected labels
- **Info**: unexpected properties not in the model

In [7]:
result = profile_to_definition(profile, graph_definition)

print("is_valid:", result.is_valid)
print(f"Errors:   {len(result.errors)}")
print(f"Warnings: {len(result.warnings)}")
print()

for issue in result.issues:
    print(f"  [{issue.severity.value}] [{issue.code}] {issue.message}")
    if issue.context:
        print(f"    context: {issue.context}")

is_valid: True
Errors:   0
Warnings: 0

  [info] [CONSTRAINT_UNVERIFIABLE] Property 'name' on City is declared required but constraint information is unavailable for this backend/strategy profile
  [info] [UNEXPECTED_PROPERTY] Property 'country' on City found in profile but not in model
  [info] [UNEXPECTED_PROPERTY] Property 'rating' on Movie found in profile but not in model
  [info] [CONSTRAINT_UNVERIFIABLE] Property 'title' on Movie is declared required but constraint information is unavailable for this backend/strategy profile
  [info] [UNEXPECTED_PROPERTY] Property 'age' on Person found in profile but not in model
  [info] [UNEXPECTED_PROPERTY] Property 'email' on Person found in profile but not in model
  [info] [CONSTRAINT_UNVERIFIABLE] Property 'name' on Person is declared required but constraint information is unavailable for this backend/strategy profile
  [info] [CONSTRAINT_UNVERIFIABLE] Property 'role' on Person:ACTED_IN:Movie is declared required but constraint informatio

The validation should report:

- A **warning** that `age` on `Person` is only 67% complete (it's a required property in the model)
- The result is still `is_valid: True` because incomplete required properties generate warnings, not errors -- the type exists and data is mostly there, but quality is imperfect

## Convert the model schema to a NetworkX graph

The `schema_to_networkx` function converts the model definition itself -- not instance data -- into a NetworkX `MultiDiGraph`. Nodes in this graph represent node types; edges represent relationship types. This is useful for programmatic analysis of the schema structure (topology, path analysis, etc.).

In [8]:
schema_graph = schema_to_networkx(graph_definition)

print("Schema graph nodes:")
for node, attrs in schema_graph.nodes(data=True):
    print(f"  {node}: uid_field={attrs['uid_field']}, properties={attrs['properties']}")

print()
print("Schema graph edges:")
for src, tgt, attrs in schema_graph.edges(data=True):
    print(f"  {src} --[{attrs['label']}]--> {tgt}")
    print(
        f"    source_cardinality={attrs['source_cardinality']}, target_cardinality={attrs['target_cardinality']}"
    )

Schema graph nodes:
  Person: uid_field=name, properties={'name': 'str', 'born': 'int'}
  Movie: uid_field=title, properties={'title': 'str', 'released': 'int', 'year': 'int'}
  City: uid_field=name, properties={'name': 'str'}

Schema graph edges:
  Person --[ACTED_IN]--> Movie
    source_cardinality=min=0 max=None, target_cardinality=min=0 max=None
  Person --[DIRECTED]--> Movie
    source_cardinality=min=0 max=None, target_cardinality=min=0 max=None
  Person --[LIVES_IN]--> City
    source_cardinality=min=0 max=None, target_cardinality=min=0 max=None


## Serialise the profile

Since `GraphProfile` is a frozen Pydantic model, it can be serialised to JSON. This is useful for storing profiles as CI artifacts, comparing profiles across time, or feeding them into downstream tools.

In [9]:
profile_json = profile.model_dump_json(indent=2)
print(profile_json[:500], "\n...")

{
  "source": "networkx",
  "timestamp": "2026-06-27T03:04:08.046213",
  "node_type_profiles": {
    "City": {
      "label": "City",
      "count": 1,
      "property_profiles": {
        "country": {
          "name": "country",
          "present_count": 1,
          "total_count": 1,
          "constraint_required": null,
          "observed_types": [
            "str"
          ],
          "observed_type_counts": {},
          "distinct_count": null,
          "value_distribution": null,
  
...
